# Semantics Continued Pretraining (DAPT) & Corpus Building Pipeline

This notebook runs the Semantics Layer pipeline directly in Google Colab.

### Assumptions:
- Your project folder (containing `data`, `evals`, `lib`, and `models`) is stored in your Google Drive.
- You have a GPU runtime enabled in Colab (highly recommended for CPT step `s2`).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure Directory Path and Change Working Directory

Set `GOOGLE_DRIVE_PATH` to the location of the project folder in your Google Drive.

In [ ]:
import os
import sys

# Adjust this path to the location of your project folder in Google Drive
GOOGLE_DRIVE_PATH = "/content/drive/MyDrive/Semantics"

if os.path.exists(GOOGLE_DRIVE_PATH):
    os.chdir(GOOGLE_DRIVE_PATH)
    sys.path.append(GOOGLE_DRIVE_PATH)
    print(f"Successfully changed working directory to: {os.getcwd()}")
else:
    print(f"Error: Google Drive path '{GOOGLE_DRIVE_PATH}' does not exist. Please check your path.")

## 3. Install Dependencies

Install the required libraries on the Colab runtime environment.

In [ ]:
# Install dependencies required by Docling and Continued Pretraining (CPT/DAPT)
!pip install -q docling python-dotenv transformers accelerate torch boto3 tiktoken hf-xet pydrive2

## 4. Load Environment Variables & Handle Path Overrides

Loads settings from `.env`. If running in Colab (Linux), it dynamically overrides absolute Windows paths in `.env` with platform-neutral relative paths.

In [ ]:
from dotenv import load_dotenv

# Load configurations from .env file
load_dotenv()

# Override absolute Windows paths to relative paths if on Colab/Linux
if os.name != 'nt':
    print("Detected Linux/Colab environment. Overriding absolute Windows paths with cross-platform relative paths...")
    os.environ["LOCAL_DIRECTORY_PATH"] = "./data/raw"
    os.environ["OUTPUT_PATH"] = "./data/dapt/domain_dapt_corpus.jsonl"
    os.environ["PROBE_QA_PATH"] = "./evals/dapt/probe_qa.jsonl"

print("\nCurrent Environment Configurations:")
print(f"STORAGE_TARGET: {os.getenv('STORAGE_TARGET')}")
print(f"LOCAL_DIRECTORY_PATH: {os.getenv('LOCAL_DIRECTORY_PATH')}")
print(f"OUTPUT_PATH: {os.getenv('OUTPUT_PATH')}")
print(f"PROBE_QA_PATH: {os.getenv('PROBE_QA_PATH')}")
print(f"BASE_MODEL_NAME: {os.getenv('BASE_MODEL_NAME')}")
print(f"DAPT_EPOCHS: {os.getenv('DAPT_EPOCHS')}")
print(f"DAPT_LR: {os.getenv('DAPT_LR')}")
print(f"DAPT_BATCH_SIZE: {os.getenv('DAPT_BATCH_SIZE')}")
print(f"CHUNK_SIZE: {os.getenv('CHUNK_SIZE')}")

In [ ]:
import time
def calculate_time(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()
        print(f"Function '{func.__name__}' took {end - start:.6f} seconds to run.")
        return result
    return wrapper


## 5. Step 1: Corpus Construction (s1)

Extracts and parses raw PDF documents from `data/raw` to build the pretraining corpus.

In [ ]:
from lib.s1_build_corpus import run_corpus_builder

@calculate_time
def s1():
    output_path = os.getenv("OUTPUT_PATH", "./data/dapt/domain_dapt_corpus.jsonl")
    storage_target = os.getenv("STORAGE_TARGET", "windows")
    local_directory_path = os.getenv("LOCAL_DIRECTORY_PATH", "./data/raw")
    aws_bucket_name = os.getenv("AWS_BUCKET_NAME")
    aws_prefix = os.getenv("AWS_PREFIX", "")
    gdrive_folder_id = os.getenv("GDRIVE_FOLDER_ID")
    available_gpus = os.getenv("AVAILABLE_GPUS", "0")
    workers_per_gpu = int(os.getenv("WORKERS_PER_GPU", "1"))
    try:
        chunk_size = int(os.getenv("CHUNK_SIZE", "10"))
    except ValueError:
        chunk_size = 10

    print("Starting Step 1 (Corpus Construction)...")
    run_corpus_builder(
        output_path=output_path,
        storage_target=storage_target,
        local_directory_path=local_directory_path,
        aws_bucket_name=aws_bucket_name,
        aws_prefix=aws_prefix,
        gdrive_folder_id=gdrive_folder_id,
        available_gpus=available_gpus,
        workers_per_gpu=workers_per_gpu,
        chunk_size=chunk_size,
    )
    print("Step 1 completed successfully!")

s1()

## 6. Step 2: Continued Pretraining (DAPT) & Evaluation (s2)

Performs domain adaptive pretraining (DAPT) on the base model using the chunked training blocks. Evaluates perplexity and QA accuracy before and after CPT.

In [ ]:
from lib.s2_dapt import run_dapt_pipeline

@calculate_time
def s2():
    model_name = os.getenv("BASE_MODEL_NAME", "HuggingFaceTB/SmolLM2-135M")
    corpus_path = os.getenv("OUTPUT_PATH", "./data/dapt/domain_dapt_corpus.jsonl")
    probe_qa_path = os.getenv("PROBE_QA_PATH", "./evals/dapt/probe_qa.jsonl")
    output_dir = os.getenv("DAPT_OUTPUT_DIR", "./models/dapt_model")
    epochs = int(os.getenv("DAPT_EPOCHS", "3"))
    lr = float(os.getenv("DAPT_LR", "5e-5"))
    batch_size = int(os.getenv("DAPT_BATCH_SIZE", "2"))

    print("Starting Step 2 (DAPT continued pretraining & evaluation)...")
    run_dapt_pipeline(
        model_name=model_name,
        corpus_path=corpus_path,
        probe_qa_path=probe_qa_path,
        epochs=epochs,
        lr=lr,
        batch_size=batch_size,
        output_dir=output_dir,
    )
    print("Step 2 completed successfully!")

s2()